# Milestone 6 Lab, notebook 4 of 8: Run health and restatements

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| **4. Run health and restatements** (you are here) | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **about 6**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

Two things, both about explaining what happened.

1. **Run health.** A run now loads, checks, sets aside, calculates, proves, and saves. When something is slow or wrong, which step was it? Each run records that.
2. **Restatements, explained by AI.** The leader was told a figure at close. It has changed. The analyst must give the true reason, from the records.

> This notebook needs `OPENROUTER_API_KEY` in your `.env` file. Run notebook 1 first.

In [ ]:
import pandas as pd
from lab_helpers import LAB_E2E, health_table, load_result, show_answer

from retail_profit_stack.e2e import build_everything, open_stores

workspace = build_everything(LAB_E2E)
stores = open_stores(workspace)
restated_key = load_result("answer_key_restated", required_from="notebook 1")

`build_everything` is notebooks 1 and 2 in one call: it builds to month-end close, loads the late files, and makes `RUN-0003` with an incremental run.

`open_stores` gives two stores. A **store** is what the analyst's tools read from. One reads the run at close, and one reads the run after the late files.

In [ ]:
for stage, store in stores.items():
    print(f"{stage:6} reads {store.run_id}, and compares it with {store.previous_run_id}")

## Step 1: A run's health

Every run record now has a `health` part: one line per step, with how long it took and how many rows it handled.

In [ ]:
from retail_profit_stack.lineage.runs import read_history

history = read_history(workspace.warehouse)
latest = history[-1]

health_table(latest)

What to look at:

- **seconds.** The whole pipeline takes a fraction of a second on this much data. With real data, this column is where you look first when a run is slow.
- **rows.** The number that moved through each step. A step that suddenly handles far fewer rows than yesterday is a warning sign, even when nothing failed.

A run also lists its **warnings**: things a person should know that did not stop the run.

In [ ]:
for record in history:
    print(record["run_id"], f"{record['mode']:12}", record["months_recalculated"], record["health"]["warnings"])

Read down the list. The first run set nothing aside, so all seven findings were waiting. The second quarantined the duplicates, leaving four. The third was incremental, and has three waiting: the missing cost arrived, so that finding is gone.

All three runs side by side, to compare the time of each step:

In [ ]:
steps = pd.DataFrame({record["run_id"]: {step["step"]: step["seconds"] for step in record["health"]["steps"]}
                      for record in history})
steps

## Step 2: The same question, two true answers

Here is a question the leader might ask twice.

In [ ]:
QUESTION = "Why did the trail water bottle's October 2026 contribution profit change since the last run?"

Asked at close, the true answer is the one from milestone 4: duplicated records were set aside. Asked after the late files, the true answer is different: a shipments file was corrected.

The run comparison tool knows both, because it compares the run it reads with the run before it. Call it yourself first, with no model.

In [ ]:
from retail_profit_stack import tools

for stage, store in stores.items():
    comparison = tools.compare_runs("2026-10", "trail water bottle", store=store)
    figure = comparison.figures_changed[0]
    print(stage, ":", figure.earlier, "->", figure.later, "  change", figure.change)
    print("   settings changed:", comparison.settings_changed)
    print("   files corrected: ", comparison.files_corrected)

Now ask the analyst, in the state after the late files.

In [ ]:
from retail_profit_stack.agent import run_analyst

late_answer = run_analyst(QUESTION, store=stores["late"])

show_answer(late_answer.answer)

Check it against your answer key from notebook 1, and against the verifier.

In [ ]:
key = restated_key["P001 2026-10"]
said = {figure.kind: figure.value for figure in late_answer.answer.figures}

print("Before: by hand", key["before"], "  the analyst said", said.get("earlier_run_value"))
print("After:  by hand", key["after"], "  the analyst said", said.get("value"))
print("Tools called:", [call.name for call in late_answer.tool_log])
print("Verified:", late_answer.verification.passed, "  Run named:", late_answer.answer.run_id)

If a check failed, run the cell that asks the analyst again.

Did the answer name the corrected shipments file, and only that file? Three files arrived late, and only one touched this product. The tool returns only the files whose records changed for the product asked about. The analyst did not work the reason out. Code found it, and the analyst put it into words.

Now the same words, asked at close.

In [ ]:
close_answer = run_analyst(QUESTION, store=stores["close"])

print(close_answer.answer.headline)
print("Run named:", close_answer.answer.run_id)

A different figure, a different reason, a different run. Both answers are true, each for its own moment. That is why every answer names its run.

## Step 3: A figure that was not there before

In [ ]:
stove = run_analyst(
    "Can we report the camp stove's October 2026 contribution profit now, and what changed since the last run?",
    store=stores["late"],
)

show_answer(stove.answer)

In [ ]:
print("By hand in notebook 1:", restated_key["P007 2026-10"]["after"])
print("Verified:", stove.verification.passed)

## Step 4: Is the explanation right?

Look at what we just checked. Code checked the **figures**, the **run**, and the **warnings**. Nothing checked the **reason**.

An answer could give the right three numbers and then say "this changed because of seasonal demand". Every check would pass. Only a reader would notice.

Whether an explanation gives the right reason is a judgement. That is the next notebook.

## What we now know

- **Run health** records each step's time and row count, and any warning. It is the first place to look when a run is slow or odd.
- The same question can have two true answers at two moments. The run named in the answer says which.
- The analyst explains a **restatement** from the run comparison: the figures, and the files that caused the change.
- Code checks figures, runs, and warnings. It cannot check whether a **reason** is right.

**Next:** notebook 5, where a second AI model judges the analyst's reasons.